In [1]:
import os
from typing import Iterable
import altair as alt
import numpy as np
import pandas as pd
import requests
import eco_style
from pyproj import Transformer
from shapely.ops import unary_union
import geopandas as gpd
import json


alt.theme.enable("light")

ThemeRegistry.enable('light')

In [2]:
import sys
sys.path.append('/Users/student/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/1. LSE/Postgraduate/Growth Lab/Chart of the Day/Weather')

from eco_style import pallete, alt

In [3]:
# Fetch daily max and min temps for London, Jan 1940 – present
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 51.5085,
    "longitude": -0.1257,
    "start_date": "1940-01-01",
    "end_date": "2026-05-25",
    "daily": ["temperature_2m_max", "temperature_2m_min"],
    "timezone": "Europe/London"
}

response = requests.get(url, params=params)
data = response.json()

# Build dataframe
df = pd.DataFrame({
    "date": pd.to_datetime(data["daily"]["time"]),
    "tmax": data["daily"]["temperature_2m_max"],
    "tmin": data["daily"]["temperature_2m_min"],
})

# Filter to May only
df_may = df[df["date"].dt.month == 5].copy()
df_may["year"] = df_may["date"].dt.year

# Average tmax and tmin by year
may_annual = df_may.groupby("year")[["tmax", "tmin"]].mean().reset_index()
may_annual.columns = ["year", "mean_tmax", "mean_tmin"]

print(may_annual.head())
print(may_annual.tail())
print(f"\n{len(may_annual)} years of data")

   year  mean_tmax  mean_tmin
0  1940  18.341935   8.890323
1  1941  13.300000   6.009677
2  1942  15.167742   7.509677
3  1943  17.367742   8.264516
4  1944  17.235484   6.287097
    year  mean_tmax  mean_tmin
82  2022  18.316129   9.351613
83  2023  17.200000   8.077419
84  2024  18.403226   9.790323
85  2025  19.903226  10.435484
86  2026  19.076000  10.164000

87 years of data


In [4]:
# Fetch daily mean temp for London, 2006–2026
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 51.5085,
    "longitude": -0.1257,
    "start_date": "2006-01-01",
    "end_date": "2026-05-25",
    "daily": ["temperature_2m_max", "temperature_2m_min", "temperature_2m_mean"],
    "timezone": "Europe/London"
}

response = requests.get(url, params=params)
data = response.json()

df_daily = pd.DataFrame({
    "date": pd.to_datetime(data["daily"]["time"]),
    "tmax": data["daily"]["temperature_2m_max"],
    "tmin": data["daily"]["temperature_2m_min"],
    "tmean": data["daily"]["temperature_2m_mean"],
})

# Filter to May only, add day-of-month and year
df_may_daily = df_daily[df_daily["date"].dt.month == 5].copy()
df_may_daily["year"] = df_may_daily["date"].dt.year
df_may_daily["day"] = df_may_daily["date"].dt.day


In [5]:
print(df_may_daily.head(10))
print(f"\n{df_may_daily['year'].nunique()} years, {len(df_may_daily)} rows")

          date  tmax  tmin  tmean  year  day
120 2006-05-01  13.9   7.4   10.4  2006    1
121 2006-05-02  15.2   3.2   10.3  2006    2
122 2006-05-03  17.8   7.0   12.9  2006    3
123 2006-05-04  24.0  10.1   17.3  2006    4
124 2006-05-05  18.9  11.0   14.9  2006    5
125 2006-05-06  16.7   7.0   11.9  2006    6
126 2006-05-07  16.5   7.9   12.5  2006    7
127 2006-05-08  15.4   9.3   12.2  2006    8
128 2006-05-09  18.8   6.7   12.5  2006    9
129 2006-05-10  20.9   8.4   14.7  2006   10

21 years, 645 rows


In [6]:
# Long-run average tmax for each calendar day in May (1940–2024)
baseline = df_full_may[df_full_may["year"] <= 2024].groupby("day")["tmax"].mean().reset_index()
baseline.columns = ["day", "baseline_tmax"]

# 2026 only
may_2026 = df_full_may[df_full_may["year"] == 2026][["day", "tmax"]].copy()

# Merge and calculate anomaly
anomaly = may_2026.merge(baseline, on="day")
anomaly["anomaly"] = anomaly["tmax"] - anomaly["baseline_tmax"]

print(anomaly)

NameError: name 'df_full_may' is not defined

In [ ]:
# Ensure correct types
df_full_may = df_full_may.copy()
df_full_may["year"] = df_full_may["year"].astype(int)
df_full_may["day"] = df_full_may["date"].dt.day

# 3-day smoothing within each year
df_full_may["tmax_smooth"] = (
    df_full_may.groupby("year")["tmax"]
    .transform(lambda s: s.rolling(3, center=True).mean())
)

climatology = (
    df_full_may[df_full_may["year"] < 2021]
    .groupby("day", as_index=False)["tmax"]
    .mean()
    .rename(columns={"tmax": "clim_tmax"})
)

recent = df_full_may[df_full_may["year"].isin([2022, 2023, 2024, 2025])]
y2026 = df_full_may[df_full_may["year"] == 2026]

In [ ]:
print(last_5[last_5["year"] == 2026][["day", "tmax", "year_str"]].head(10))
print(last_5["year_str"].unique())

     day  tmax year_str
124    1  24.1     2026
125    2  22.5     2026
126    3  19.2     2026
127    4  16.3     2026
128    5  16.5     2026
129    6  13.3     2026
130    7  15.0     2026
131    8  18.5     2026
132    9  21.0     2026
133   10  13.6     2026
<ArrowStringArray>
['2022', '2023', '2024', '2025', '2026']
Length: 5, dtype: str


In [ ]:
print(last_5_smooth["year"].unique())
print(last_5_smooth["year_str"].unique())
print(last_5_smooth.groupby("year")["day"].count())

[2022 2023 2024 2025 2026]
<ArrowStringArray>
['2022', '2023', '2024', '2025', '2026']
Length: 5, dtype: str
year
2022    31
2023    31
2024    31
2025    31
2026    25
Name: day, dtype: int64


In [ ]:
def smooth(series, window=5):
    return series.rolling(window=window, center=True, min_periods=1).mean()

# Smooth all years
all_smoothed_frames = []
for year, group in df_full_may.groupby("year"):
    group = group.sort_values("day").copy()
    group["tmax_smooth"] = smooth(group["tmax"])
    all_smoothed_frames.append(group)

df_all_smooth = pd.concat(all_smoothed_frames)

# Last point for each year for dots and labels
all_last_points = df_all_smooth.loc[df_all_smooth.groupby("year_str")["day"].idxmax()].reset_index(drop=True)

print(f"{df_all_smooth['year'].nunique()} years smoothed")
print(all_last_points[["year", "day", "tmax_smooth"]].tail())

87 years smoothed
    year  day  tmax_smooth
82  2022   31    14.900000
83  2023   31    17.366667
84  2024   31    17.333333
85  2025   31    25.466667
86  2026   25    32.000000


In [ ]:
color_scale_all = alt.Scale(
    domain=["2025", "2026", "other"],
    range=[pallete["nominal_1"], pallete["nominal_2"], "rgba(24, 42, 56, 0.15)"]
)

df_all_smooth["year_cat"] = df_all_smooth["year"].apply(
    lambda y: str(y) if y >= 2025 else "other"
)
all_last_points["year_cat"] = all_last_points["year"].apply(
    lambda y: str(y) if y >= 2025 else "other"
)

# All years except 2025/2026 — very muted
grey_lines = alt.Chart(df_all_smooth[df_all_smooth["year"] < 2025]).mark_line(
    strokeWidth=0.8,
    opacity=0.15,
    color="#676A86",
).encode(
x=alt.X("day:Q",
        scale=alt.Scale(domain=[1, 31]),
        axis=alt.Axis(tickCount=8, labelExpr="'May ' + datum.value", format="d"),
        title=""),
    y=alt.Y("tmax_smooth:Q",
            scale=alt.Scale(zero=False),
            axis=alt.Axis(labelExpr="datum.value + '°C'"),
            title=""),
    detail="year_str:N"
)

# 2025 
line_2025 = alt.Chart(df_all_smooth[df_all_smooth["year"] == 2025]).mark_line(
    strokeWidth=2.5,
    color="#9EB3C2"
).encode(
x=alt.X("day:Q",
        scale=alt.Scale(domain=[1, 31]),
        axis=alt.Axis(tickCount=8, labelExpr="'May ' + datum.value", format="d"),
        title=""),
    y=alt.Y("tmax_smooth:Q"),
    tooltip=[
        alt.Tooltip("day:O", title="May"),
        alt.Tooltip("tmax:Q", title="2025 max (°C)", format=".1f")
    ]
)

# 2026 red
line_2026 = alt.Chart(df_all_smooth[df_all_smooth["year"] == 2026]).mark_line(
    strokeWidth=2.5,
    color=pallete["nominal_2"]
).encode(
x=alt.X("day:Q",
        scale=alt.Scale(domain=[1, 31]),
        axis=alt.Axis(tickCount=8, labelExpr="'May ' + datum.value", format="d"),
        title=""),
    y=alt.Y("tmax_smooth:Q"),
    tooltip=[
        alt.Tooltip("day:O", title="May"),
        alt.Tooltip("tmax:Q", title="2026 max (°C)", format=".1f")
    ]
)

# Dots for 2025 and 2026 only
highlight_last = all_last_points[all_last_points["year"] >= 2025]

end_dots = alt.Chart(highlight_last).mark_point(
    filled=True, size=70
).encode(
x=alt.X("day:Q",
        scale=alt.Scale(domain=[1, 31]),
        axis=alt.Axis(tickCount=8, labelExpr="'May ' + datum.value", format="d"),
        title=""),
    y=alt.Y("tmax_smooth:Q"),
    color=alt.Color("year_cat:N",
                    scale=alt.Scale(
                        domain=["2025", "2026"],
                        range=[ "#9EB3C2", pallete["nominal_2"]]
                    ), legend=None)
)

# Labels for 2025 and 2026 only
end_labels = alt.Chart(highlight_last).mark_text(
    align="left", dx=6, fontSize=11, fontWeight="bold"
).encode(
    x=alt.X("day:Q",
        scale=alt.Scale(domain=[1, 31]),
        axis=alt.Axis(tickCount=8, labelExpr="'May ' + datum.value", format="d"),
        title=""),
    y=alt.Y("tmax_smooth:Q"),
    text="year_str:N",
    color=alt.Color("year_cat:N",
                    scale=alt.Scale(
                        domain=["2025", "2026"],
                        range=[ "#9EB3C2", pallete["nominal_2"]]
                    ), legend=None)
)

chart = (grey_lines + line_2025 + line_2026 + end_dots + end_labels).properties(
    width=620,
    height=400,
    title=alt.TitleParams(
        text="London's May 2026 has been the hottest on record",
        subtitle=[
            "Daily maximum temperature in May, London, 1940–2026. Each line is one year.",
            "Source: Open-Meteo Historical Weather API"
        ],
        anchor="start"
    )
)

chart

alt.LayerChart(...)

In [ ]:
import pandas as pd

bh = pd.read_csv('/Users/student/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/1. LSE/Postgraduate/Growth Lab/Chart of the Day/Weather/late_may_bank_holiday_temperatures.csv')

bh["Date"] = pd.to_datetime(bh["Date"])
bh["year"] = bh["Date"].dt.year
bh["color_cat"] = bh["year"].apply(lambda y: "2026" if y == 2026 else "other")

mean_val = bh[bh["year"] < 2026]["Value"].mean()

bars = alt.Chart(bh).mark_bar(width=3).encode(
x=alt.X("year:O", 
        axis=alt.Axis(
            values=[1880, 1900, 1920, 1940, 1960, 1980, 2000, 2020, 2026],
            labelExpr="datum.value == 2026 ? '' : datum.value",
            labelAngle=0,
            title=""
        )),
    y=alt.Y("Value:Q",
            axis=alt.Axis(labelExpr="datum.value + '°C'"),
            title=""),
    color=alt.condition(
        alt.datum.year == 2026,
        alt.value(pallete["nominal_2"]),
        alt.value(pallete["nominal_1"])
    ),
    tooltip=[
        alt.Tooltip("year:O", title="Year"),
        alt.Tooltip("Date:T", title="Date", format="%d %b %Y"),
        alt.Tooltip("Value:Q", title="Max temp (°C)", format=".1f")
    ]
)

mean_line = alt.Chart(pd.DataFrame({"y": [mean_val]})).mark_rule(
    strokeDash=[4, 3],
    strokeWidth=1.2,
    color=pallete["domain"],
    opacity=0.6
).encode(y="y:Q")

mean_label = alt.Chart(pd.DataFrame({
    "y": [mean_val],
    "x": [2026],
    "text": [f"Average: {mean_val:.1f}°C"]
})).mark_text(
    align="right", dx=-4, dy=-8, fontSize=10,
    color=pallete["domain"]
).encode(
    x=alt.value(620),
    y="y:Q",
    text="text:N"
)

# Label for 2026
label_2026 = alt.Chart(bh[bh["year"] == 2026]).mark_text(
    align="center",
    dy=-12,
    fontSize=11,
    fontWeight="bold",
    color=pallete["nominal_2"]
).encode(
    x=alt.X("year:O"),
    y=alt.Y("Value:Q"),
    text=alt.Text("Value:Q", format=".1f")
)

# Label for 1944
label_1944 = alt.Chart(bh[bh["year"] == 1944]).mark_text(
    align="center",
    dy=-12,
    fontSize=10,
    color=pallete["nominal_1"]
).encode(
    x=alt.X("year:O"),
    y=alt.Y("Value:Q"),
    text=alt.value("Previous record: 28.9°C (1944)")
)

chart = (bars + mean_line + label_2026 + label_1944).properties(
    width=650,
    height=380,
    title=alt.TitleParams(
        text="This bank holiday is the hottest on record",
        subtitle=[
            "Maximum temperature on the late May bank holiday, Central England, 1878–2026",
            "Source: Met Office"
        ],
        anchor="start"
    )
)

chart

alt.LayerChart(...)

In [9]:
# Fetch daily tmax for London from Open-Meteo 1940–present
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 51.5085,
    "longitude": -0.1257,
    "start_date": "1940-01-01",
    "end_date": "2026-05-25",
    "daily": ["temperature_2m_max"],
    "timezone": "Europe/London"
}

response = requests.get(url, params=params)
data = response.json()

df_om = pd.DataFrame({
    "date": pd.to_datetime(data["daily"]["time"]),
    "tmax": data["daily"]["temperature_2m_max"],
})

# Load bank holiday dates from your CSV
bh_dates = pd.read_csv('/Users/student/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/1. LSE/Postgraduate/Growth Lab/Chart of the Day/Weather/late_may_bank_holiday_temperatures.csv')
bh_dates["Date"] = pd.to_datetime(bh_dates["Date"])

# Filter Open-Meteo data to just the bank holiday dates from 1940 onwards
bh_om = df_om[df_om["date"].isin(bh_dates["Date"])].copy()
bh_om["year"] = bh_om["date"].dt.year

print(bh_om.head())
print(f"\n{len(bh_om)} bank holidays matched")

           date  tmax  year
147  1940-05-27  18.8  1940
511  1941-05-26  13.4  1941
875  1942-05-25  13.4  1942
1246 1943-05-31  19.8  1943
1610 1944-05-29  29.5  1944

87 bank holidays matched


In [48]:
bh_om["year_str"] = bh_om["year"].astype(str)

mean_val = bh_om[bh_om["year"] < 2026]["tmax"].mean()

bars = alt.Chart(bh_om).mark_bar(width=4).encode(
    x=alt.X("year:O",
            axis=alt.Axis(
                values=[1940, 1960, 1980, 2000, 2020],
                labelExpr="datum.value == 2026 ? '' : datum.value",
                labelAngle=0,
                title=""
            )),
    y=alt.Y("tmax:Q",
            axis=alt.Axis(labelExpr="datum.value + '°C'"),
            title=""),
    color=alt.condition(
        alt.datum.year == 2026,
        alt.value(pallete["nominal_2"]),
        alt.value(pallete["nominal_1"])
    ),
    tooltip=[
        alt.Tooltip("year:O", title="Year"),
        alt.Tooltip("date:T", title="Date", format="%d %b %Y"),
        alt.Tooltip("tmax:Q", title="Max temp (°C)", format=".1f")
    ]
)

mean_line = alt.Chart(pd.DataFrame({"y": [mean_val]})).mark_rule(
    strokeDash=[4, 3],
    strokeWidth=1.2,
    color=pallete["domain"],
    opacity=0.6
).encode(y="y:Q")

mean_label = alt.Chart(pd.DataFrame({
    "y": [mean_val],
    "text": [f"Average: {mean_val:.1f}°C"]
})).mark_text(
    align="right", dx=-4, dy=-8, fontSize=10,
    color=pallete["domain"]
).encode(
    x=alt.value(900),
    y="y:Q",
    text="text:N"
)

# 2026 label
label_2026 = alt.Chart(bh_om[bh_om["year"] == 2026]).mark_text(
    align="center", dy=-12, fontSize=11,
    fontWeight="bold", color=pallete["nominal_2"]
).encode(
    x=alt.X("year:O"),
    y=alt.Y("tmax:Q"),
    text=alt.Text("tmax:Q", format=".1f")
)

# 1944 label
label_1944 = alt.Chart(bh_om[bh_om["year"] == 1944]).mark_text(
    align="center", dy=-45, fontSize=10,
    color=pallete["nominal_1"]
).encode(
    x=alt.X("year:O"),
    y=alt.Y("tmax:Q"),
    text=alt.value(["Previous", "record: ", "29.5°C ", "(1944)"])
)

bar = (bars + mean_line + label_2026 + label_1944).properties(
    width=600,
    height=380,
    title=alt.TitleParams(
        text="This bank holiday is the hottest on record",
        subtitle=[
            "Maximum temperature on the late May bank holiday, London, 1940–2026",
            "Source: Open-Meteo Historical Weather API"
        ],
        subtitleColor="#676A86",
        anchor="start"
    )
)

bar

alt.LayerChart(...)

🌡️ London's late May bank holiday has never been so hot. In 87 years of records, none have come close. London hit 33.8°C on 25 May 2026. That is 4.3°C above the previous record set in 1944. 

#ChartOfTheDay

🔗 Visit our Data Hub to explore and create your own charts. https://economicsobservatory.com/explore 

In [49]:

bh_om["year_str"] = bh_om["year"].astype(str)

mean_val = bh_om[bh_om["year"] < 2026]["tmax"].mean()

bars = alt.Chart(bh_om).mark_point(filled=True, size=50, opacity=0.8).encode(
    x=alt.X("year:O",
            axis=alt.Axis(
                values=[1940, 1960, 1980, 2000, 2020],
                labelExpr="datum.value == 2026 ? '' : datum.value",
                labelAngle=0,
                title=""
            )),
    y=alt.Y("tmax:Q",
            scale=alt.Scale(zero=False, domainMax=35),
            axis=alt.Axis(labelExpr="datum.value + '°C'"),
            title=""),
    color=alt.condition(
        alt.datum.year == 2026,
        alt.value(pallete["nominal_2"]),
        alt.value(pallete["nominal_1"])
    ),
    tooltip=[
        alt.Tooltip("year:O", title="Year"),
        alt.Tooltip("date:T", title="Date", format="%d %b %Y"),
        alt.Tooltip("tmax:Q", title="Max temp (°C)", format=".1f")
    ]
)

mean_line = alt.Chart(pd.DataFrame({"y": [mean_val]})).mark_rule(
    strokeDash=[4, 3],
    strokeWidth=1.2,
    color=pallete["domain"],
    opacity=0.6
).encode(y="y:Q")

mean_label = alt.Chart(pd.DataFrame({
    "y": [mean_val],
    "text": [f"Average: {mean_val:.1f}°C"]
})).mark_text(
    align="right", dx=-4, dy=-8, fontSize=10,
    color=pallete["domain"]
).encode(
    x=alt.value(900),
    y="y:Q",
    text="text:N"
)

# 2026 label
label_2026 = alt.Chart(bh_om[bh_om["year"] == 2026]).mark_text(
    align="center", dy=-36, fontSize=11,
    fontWeight="bold", color=pallete["nominal_2"]
).encode(
    x=alt.X("year:O"),
    y=alt.Y("tmax:Q"),
    text=alt.value(["25 May", "2026: ", "33.8°C "])
)

# 1944 label
label_1944 = alt.Chart(bh_om[bh_om["year"] == 1944]).mark_text(
    align="center", dy=-50, fontSize=11,
    color=pallete["nominal_1"]
).encode(
    x=alt.X("year:O"),
    y=alt.Y("tmax:Q"),
    text=alt.value(["Previous", "record: ", "29.5°C ", "(1944)"])
)

line = alt.Chart(bh_om).mark_line(
    strokeWidth=2.0,
    opacity=0.3,
    color=pallete["nominal_1"]
).encode(
    x=alt.X("year:O"),
    y=alt.Y("tmax:Q")
)

bank_hols = (bars  + label_2026 + label_1944 ).properties(
    width=600,
    height=380,
    title=alt.TitleParams(
        text="This week's bank holiday was the hottest on record",
        subtitle=[
            "Maximum temperature on the late May bank holiday, London, 1940–2026",
            "Source: Open-Meteo Historical Weather API"
        ],
        subtitleColor="#676A86",
        anchor="start"
    )
)

bank_hols

alt.LayerChart(...)

In [50]:
bank_hols.save("bank_hols.png", scale_factor=3.0)
bank_hols.save("bank_hols.json")